<h1 align="center">Senyals i Sistemes - Pràctica 3<h1/>

## Instruccions bàsiques

L'enunciat consta d'una sèrie d'explicacions, fragments de codi Python pre-programat i també d'un conjunt de <span style="color:red; font-size:16px;">Qüestions</span>. Aquestes qüestions estan relacionades amb el __Qüestionari Atenea__ associat a la pràctica.

  1. Haureu de tenir el __qüestionari obert en paral·lel__ amb aquest document de JupyterLab.
  2. Algunes preguntes demanen __dades inicials__. Trobareu aquestes dades a la pregunta corresponent del qüestionari. A cada usuari se li assignen valors diferents.
  3. En les zones de codi que estan en blanc, __programareu la vostra solució__ a la qüestió que se us planteja, i __consignareu els resultats__ a les caselles corresponents del qüestionari.
  4. En acabar, no oblideu de __tancar__ i enviar el qüestionari per tal que sigui corregit.

## <span style="color:red">__Qüestió 1__</span> - Identificació dels membres del grup de treball

Aneu a la pregunta 1 del qüestionari i indiqueu els membres que formen el grup de treball

# <span style="color:#BB44DD">Part 1: La Transformada de Fourier de Temps Discret (DTFT)</span>

## Introducció

Com s'ha vist a les classes de teoria, la Transformada de Fourier de Temps Discret (DTFT) d'una seqüència $x[n]$ és una funció en general **complexa**, definida sobre una variable real contínua F i que es denota per $X(F)$. La seva expressió matemàtica és:

$$X(F) = \sum_{n=-\infty}^{+\infty} x[n]e^{-j2\pi F n}$$

També és **periòdica en F**, amb període 1. Aquesta periodicitat ens permet simplificar la representació de $X(F)$, ja que n'hi ha prou amb dibuixar-la a l'interval $F \in [0,1]$.

Ara bé, malauradament en un ordinador no podem emmagatzemar i/o representar senyals continus, sinó que <span style="color:red">ens hem de conformar amb calcular el seu valor en un conjunt de punts i dibuixar aquests punts en pantalla units per rectes</span>. Evidentment, com més junts estiguin els punts més s'acostarà aquesta representació al senyal continu teòric.

En el processament del senyal existeix un algorisme **molt eficient** per calcular un conjunt de punts arbitràriament gran (\*) de la DTFT. Aquest algorisme s'anomena **FFT** (*Fast Fourier Transform*, [1], [2]) i és el que utilitzarem aquí per calcular i representar les Transformades de Fourier de les diferents seqüències.

<hr/>

(\*): L'algorisme de la FFT és realment eficient quan el nombre de punts a calcular és una **potència de 2**, però com que aquí treballarem amb un nombre de punts petit, no ens importarà escollir altres valors més propers a múltiples de 10.

[1] Si esteu interessats en comprendre l'algorisme de la FFT, en [aquest vídeo](http://y2u.be/h7apO7q16V0) trobareu una explicació excel·lent del seu funcionament, feta des d'un punt de vista poc convencional.

[2] En [aquest altre vídeo](http://y2u.be/nmgFG7PUHfo) s'explica l'apassionant història del descobriment de la FFT. 

<span style="float:left">![Funcionament de la FFT](FFT_URL_1.png "Funcionament de la FFT")</span> <span style="float:right">![Història de la FFT](FFT_URL_2.png "Història de la FFT")</span>


## Codi d'inicialització

Executeu el codi llistat a continuació. El codi importa els paquets rellevants i defineix una funció que serà útil per representar de forma conjunta una seqüència i la seva DTFT. Pareu atenció als paràmetres d'entrada d'aquesta funció, ja que l'haureu d'usar més endavant.

**NOTA IMPORTANT:** Si la instrucció `%matplotlib widget`falla, possiblemen és perquè no teniu instal·lada una versió actualitzada del Jupyter Lab o bé us falta instal·lar el paquet `ipympl`. En aquest darrer cas podeu instal·lar-lo des d'un terminal fent `conda install ipympl` o bé des de l'entorn gràfic de l'Anaconda Navigator.


In [1]:
# Importació de les biblioteques bàsiques
import numpy as np
from scipy.fft import fft, ifft

##### 3 maneres diferents de presentar les gràfiques
# INLINE: Les gràfiques es pinten just a sota la cel·la que les genera, però no són interactives. 
#         Útil per crear un PDF autocontingut amb enunciat i resultats
#
# WIDGET: Similar a l'anterior però les gràfiques són interactives i redimensionables.
#         - Si obrim moltes gràfiques simultànies es pot queixer per manca de recursos
#         - Cal instal·lar el paquet 'ipympl' (no ve per defecte) per tal que funcioni
#
#     QT: Cada gràfica es crea en una finestra a part.
#         És l'opció més flexible però també dificulta documentar els resultats
#
# DESCOMENTEU UNA DE LES TRES OPCIONS
#%matplotlib inline
#%matplotlib widget 
%matplotlib qt
import matplotlib.pyplot as plt


# Fem la mida per defecte dels plots una mica més gran
plt.rcParams['figure.figsize'] = [9.5, 6]
plt.rcParams['figure.dpi'] = 80

# Augmentem una mica el nombre de gràfics oberts permesos sense warning
plt.rcParams['figure.max_open_warning'] = 30


# Definim una funció per automatitzar el dibuix d'una seqüència conjuntament amb la seva DTFT
def pintaDTFT(x, N, fm=1, superposa_dtft=False):
    """
    
    Paràmetres
    ----------
    x   : Array NumPy 1-D de nombres reals 
          Seqüència d'entrada de la qual es vol calcular la DTFT
    N   : Enter
          Nombre de punts de la FFT. Normalment serà N > len(x)
    fm  : Enter (per defecte=1)
          Freqüència de mostratge (per pintar adequadament l'eix de freqüències)

    Retorna
    -------
    f : NumPy Array
        Vector de freqüències (longitud N)
    X : Vector de valors de la DTFT a les freqüències del vector f (longitud N)

    """
    
    fig, (ax1,ax2) = plt.subplots(2, 1, constrained_layout=True)

    # Gràfica de la seqüència
    ax1.stem(x)                        # Assumim que la separació entre punts val 1
    ax1.grid(True)
    ax1.set_title("Seqüència x[n] de L={}".format(L))
    ax1.set_ylabel("Amplitud")
    ax1.set_xlabel("Índex n")
    
    # Si hem de superposar la DTFT, calculem una FFT amb molts punts, que serà una bona aproximació a la DTFT
    if superposa_dtft:
        NDTFT = 4096
        DTFT = fft(x,NDTFT)
        fdtft = fm*np.arange(NDTFT)/NDTFT
        ax2.plot(fdtft,np.abs(DTFT),"0.8")

    # Gràfica de la Transformada de Fourier
    X = fft(x,N)                       # Càlcul de la DTFT (bé, només el seu valor en N punts) usant l'algorisme FFT
    f = fm*np.arange(N)/N              # Càlcul del vector de freqüències associat (N valors, de 0 a (N-1)/N)
    ax2.plot(f,np.abs(X),
             "C1o-" if N< 200 else "C1-",
             markersize=2.5,
             markerfacecolor="black",
             markeredgecolor="black")  # Pintem el mòdul de la FFT, unint els punts (negres) amb rectes (taronges, color "C1")
    ax2.set_xticks(fm*np.arange(21)/20)  # 20 ticks cada 0.05*fm
    ax2.grid(True)
    ax2.set_title("Transformada de Fourier X(F) amb N={}".format(N))
    ax2.set_ylabel("Amplitud")
    ax2.set_xlabel("Freqüència F")
    
    return f,X


## <span style="color:red">__Qüestió 2__</span> - Càlcul de la DTFT amb la FFT

Genereu les seqüències sinusoïdals amb la longitud i freqüència que se us indica al qüestionari, i representeu-les amb l'ajut de la funció «pintaDTFT()».

A la vista dels resultats, **responeu els apartat de la pregunta 2**.

In [20]:
# Introduïu aquí el codi i executeu-lo

# Longitud de la seqüència
L  = 10
n  = np.arange(L)              # Vector d'índex per generar el cosinus

# Cas 1:
Fo = 0.1
x  = np.cos(2*np.pi*Fo*n)                          # Cosinus

# paràmetres de la DTFT, generació i representació gràfica
N = 100                           # Longitud de la FFT
f,X = pintaDTFT(x,N, superposa_dtft = True)

# Calculeu el valor del màxim i la freqüència a la que es dóna
Xmax = max(np.abs(X))
Fmax = f[np.argmax(np.abs(X))]
print("Màxim pel cas N=L: {} a F={}".format(Xmax, Fmax))

# # Cas 2:
# Fo = 0.18
# x  = np.cos(2*np.pi*Fo*n)                          # Cosinus


Màxim pel cas N=L: 5.1250558793681025 a F=0.11


## <span style="color:red">__Qüestió 3__</span> - Conclusions

Amb els casos que acabeu de representar podeu fer-vos una idea aproximada de com podeu usar la DTFT per determinar la freqüència d'un senyal, i de quin és l'error d'estimació en funció del nombre de punts de la FFT. Tenint en ment tot això, **responeu la pregunta 3 del qüestionari**.

___

# <span style="color:#BB44DD">Part 2: Característiques freqüencials d'algunes finestres</span>

En la secció anterior hem treballat amb un segment d'un senyal sinusoïdal. En ocasions, per a millorar la detecció d'alguns paràmetres del senyal és habitual multiplicar-lo per una funció (**finestra**) d'una certa forma. Al llarg dels anys s'han proposat i investigat un elevat nombre de finestres diferents, cadascuna amb les seves característiques particulars.

El mòdul `scipy.signal.windows` conté funcions per obtenir les seqüències corresponents als diferents tipus de finestres, amb la longitud desitjada. A efectes comparatius resulta interessant pintar la forma de cada finestra juntament amb la forma de la seva DTFT. Per fer-ho podem utilitzar la mateixa funció «pintaDTFT» que s'ha usat a l'apartat anterior, però passant-li la seqüència de la finestra en lloc de la del senyal.

A continuació:

  * Obteniu les seqüències corresponents a les finestres **boxcar** (rectangular), **bartlett**, **hamming**, **blackmanharris** i **flattop**, totes elles de la longitud L que s'indica a la pregunta 4.
  * Per cada finestra, calculeu el mòdul de la seva DTFT usant una FFT de 1024 punts.
  * Pinteu cada finestra juntament amb la seva DTFT, fixant-vos en les diferències entre elles.
  * Mesureu el valor màxim de cada finestra (Mòdul en F=0)


*OPCIONAL:* Si teniu curiositat, podeu provar també altres finestres que trobareu [descrites a la documentació del SciPy](https://docs.scipy.org/doc/scipy/reference/signal.windows.html).

In [25]:
from scipy.signal import windows

L = 32        # Nombre de punts de la finestra
N = 1024        # Nombre de punts de la FFT

f,X = pintaDTFT(windows.hamming(L), N, superposa_dtft= True)
Xmax = max(np.abs(X))
Fmax = f[np.argmax(np.abs(X))]
Fmin = f[np.argmin(np.abs(X))]
print("Màxim pel cas N=L: {} a F={}".format(Xmax, Fmax))
Fmin

Màxim pel cas N=L: 16.82 a F=0.0


0.5

## <span style="color:red">__Qüestions 4 i 5__</span> - Característiques freqüencials de les finestres rectangular i de Hamming

Feu les mesures pertinents sobre les DTFT de les finestres **rectangular** i de **Hamming** que acabeu d'obtenir i responeu els apartats de les preguntes 4 i 5 del qüestionari.

Podeu fer les mesures «a ull», fent *zoom* a les zones rellevant de la gràfica, o podeu cercar els valors demanats usant instruccions de Python, a la cel·la del dessota. En qualsevol cas, no us hi entretingueu excessivament ja que amb uns valors aproximats serà suficient.

In [ ]:
###################################
# ATENCIÓ: --> APARTAT *OPCIONAL* #
###################################

# Si voleu determinar els valors demanats usant el Python, introduïu aquí el codi necessari
# Calcular el màxim és simple

# Calcular l'amplada no ho és tant. Una idea podria ser la de detectar el *primer* mínim de la TF
# i calcular la freqüència a la que correspon.
# El primer mínim el trobem derivant (np.diff): abans del mínim la derivada és negativa, i després és positiva
# Calculem el *signe* de la derivada (np.sign) i quan canvia de -1 a +1 (salt de 2) per allà tenim el mínim (np.where(condició))
# N'hi ha uns quants: ens quedem amb el primer, naturalment...

# Finestra rectangular
Maxim = 
Amplada = 

print("Finestra rectangular de L={}:".format(L))
print("    Valor màxim (amplitud): {}".format(Maxim))
print("    Amplada del lòbul principal: {}\n".format(Amplada))

# Finestra Hamming
Maxim =
Amplada = 
print("Finestra Hamming de L={}:".format(L))
print("    Valor màxim (amplitud): {}".format(Maxim))
print("    Amplada del lòbul principal: {}\n".format(Amplada))


___

# <span style="color:#BB44DD">Part 3: Efectes de la finestra - Aplicació a la detecció de components freqüencials d'un senyal</span>

En aquesta part de la pràctica el nostre objectiu serà veure com afecta el tipus de finestra que apliquem a un senyal en l'efectivitat de la detecció dels diferents components freqüencials que conformen el senyal quan aquests components tenen amplituds i/o freqüències molt propers o molt semblants entre sí.

A tal efecte usarem uns senyals predefinits que llegirem d'un fitxer.

Baixeu el fitxer «Senyals.zip» i descomprimiu-lo la carpeta de treball. Entre els nombrosos fitxers que crea (i que usarem més endavant) trobareu el fitxer «`mixture.npy`». Aquest fitxer conté 75 senyals de longitud 32 mostres cadascun. Cada senyal és una **mescla de dues sinusoides d'amplitud i freqüència desconegudes**:

$$A_1\cos\left(2\pi F_1 n\right) + A_2\cos\left(2\pi F_2 n\right)\qquad \text{amb} F_1 < F_2$$

L'objectiu és determinar l'amplitud i freqüència de cada senyal, alhora que experimentem amb 2 tipus diferents de finestra per tal de posar en evidència els avantatges i inconvenients de cadascuna.

## <span style="color:red">__Qüestions 6, 7 i 8__</span> - Anàlisi freqüencial de senyals

  * Per cadascun dels senyals que s'indiquen a les preguntes 6, 7 i 8:
  
    1. Apliqueu primer una **finestra rectangular** i representeu el mòdul de la TF (escolliu una **FFT de 1024 punts**). 
    2. Repetiu el procediment amb una **finestra de Hamming**.
    3. Responeu els apartats de la pregunta corresponent.

**IMPORTANT:** No es tracta de ser extremadament precisos amb les mesures d'amplitud i freqüència que es demana. N'hi haurà prou amb que feu «zoom» en l'àrea adequada de les gràfiques i us fixeu amb el número que surt al dessota de la gràfica quan passem el cursor per damunt de la corba.

**TAMBÉ IMPORTANT:** Per poder fer «zoom» dins la gràfica és imprescindible usar la directiva `%matplotlib widget`, cosa que ja fem al fragment de codi del principi. Tanmateix, és possible que aquesta directiva no us funcioni correctament. En aquest cas:
  1. Comproveu que teniu una versió actualitzada del vostre entorn Python i tots els paquets implicats (especialment el Jupyter Lab)
  2. Instal·leu el paquet `ipympl` si no el teniu instal·lat: feu `conda install ipympl` des d'un terminal.
  3. Reinicieu el Jupyter Lab després del pas anterior.

In [42]:
# Carregueu els senyals del fitxer en una variable
s = np.load("mixture.npy")

# Longitud de les seqüències i les FFT
L = 32
N = 1024

# Finestres
wr = windows.boxcar(L)
wh = windows.hamming(L)

# Seleccioneu la seqüència que us indica la pregunta 6
idx = 64          # Número de la primera seqüència

# Pinteu la seva DTFT
print("DTFT de la seqüència {}".format(idx))
f,X = pintaDTFT(s[idx]*wr, N, superposa_dtft= True)

Xmax = max(np.abs(X))
Fmax = f[np.argmax(np.abs(X))]
print("Màxim pel cas N=L: {} a F={}".format(Xmax, Fmax))


DTFT de la seqüència 64
Màxim pel cas N=L: 48.300857629990766 a F=0.203125


In [ ]:
# Repetiu pel senyal indicat a la pregunta 7
idx =            # Número de la segona seqüència
print("DTFT de la seqüència {}".format(idx))
...

In [ ]:
# Repetiu pel senyal indicat a la pregunta 8
idx =            # Número de la tercera seqüència
print("DTFT de la seqüència {}".format(idx))
...

___

# <span style="color:#BB44DD">Part 4: Endevina el número</span>

## Introducció

Usa l'algorisme següent (llegeix la documentació de les funcions per saber amb quins paràmetres cal invocar-les):

  1. Llegeix el fitxer d'àudio que es demana a la pregunta 9 amb la funció `scipy.io.wavfile.read`. La funció retorna les dades d'àudio i la freqüència de mostratge.
  2. Pinteu les dades en una gràfica. Observeu que el senyal consta en la seva major part de zones de silenci trencades per breus intervals de senyal.
  3. Per cadascun dels intervals de senyal:
     - Feu «zoom» de la gràfica de manera que englobi només un dels trams de senyal (un dígit DTMF)
     - Passeu el ratolí per damunt del senyal per veure les coordenades dels punts i anoteu la mostra inicial i final del tram
     - Usant les coordenades que acabeu de trobar, creeu una nova variable que contingui només el fragment de senyal desitjat
     - Calculeu la transformada de Fourier (fft) del tram usant un nombre de punts força més elevat que la longitud del senyal (N>>L)
     - Creeu un vector de freqúències adequat a la transformada que acabeu de calcular
     - Pinteu una gràfica del valor absolut de la transformada de Fourier en funció de la freqüència.
     - Fent «zoom» al gràfic obtingut, determineu les 2 freqüències que conformen el tram de senyal (pics principals).
     - Cerqueu a la taula de tons DTMF el dígit que correspon a la combinació de freqüències trobada.


## Codi d'inicialització

Executeu el codi llistat a continuació.

**NOTA:** El paquet `pyaudio` no ve instal·lat de sèrie a l'Anaconda, i l'haureu d'instal·lar **prèviament** de forma manual fent:

``` shell
conda install pyaudio
```

In [43]:
from scipy.io import wavfile    # Lectura de fitxers 'WAV'
#import pyaudio                  # Reproducció d'àudio en Python
import IPython.display as ipd   # Widget per reproduir àudio en el Jupyter

Llegim el senyal d'àudio i extraiem la informació rellevant (mostres i freqüència de mostratge)

In [45]:
# Llegim el fitxer d'àudio
fitxer = "num6.wav"            # Canvieu-lo pel fitxer indicat a la pregunta 9
fm, s = wavfile.read(fitxer)
print("Llegit fitxer amb freqüència de mostratge fm={} Hz".format(fm))
ipd.Audio(s, rate=fm, autoplay=False)                # Això permet reproduir l'àudio des del Jupyter. No és important: Ho podem obviar si no funciona.

Llegit fitxer amb freqüència de mostratge fm=4000 Hz


In [47]:
plt.figure()
t = np.arange(len(s))/fm
plt.plot(t,s)    # De cara a la selecció dels trams és millor que l'eix d'abcisses tingui l'índex n i no el temps t
plt.plot(s)
plt.grid(True)

Ara mesureu (a ull, fent zoom) els índexs inicial i final de cada tram de la seqüència corresponent a cada dígit DTMF i, per cadascun d'ells,
  1. Extraieu el tram de seqüència en una variable auxiliar
  2. Calculeu la DTFT del tram i pinteu-la en un gràfic
  3. Determineu (a ull) les dues freqüències que formen el senyal del tram.

In [54]:
inicial = 12150
final = 12325
tram =  s[inicial:final]

pintaDTFT(tram, N=1024)

(array([0.00000000e+00, 9.76562500e-04, 1.95312500e-03, ...,
        9.97070312e-01, 9.98046875e-01, 9.99023438e-01]),
 array([-1536.           -0.j        , -1042.48569609 +476.61058076j,
         -231.62248998  -67.22108285j, ...,  -383.89228881+1438.56516582j,
         -231.62248998  +67.22108285j, -1042.48569609 -476.61058076j]))

### Resultats:

  * Primer dígit: F1= , F2= -> DÍGIT= 
  * Segon dígit: F1= , F2= -> DÍGIT= 
  * Tercer dígit: F1= , F2= -> DÍGIT= 
  * Quart dígit: F1= , F2= -> DÍGIT= 
  * Cinquè dígit: F1= , F2= -> DÍGIT= 

**NÚMERO MARCAT: X-X-X-X-X**

----

<h1 align="center">Final de l'enunciat<h1/>

----